In [0]:
# Ler o CSV com Spark

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/architect_company/churn_prediction/1_raw/customer_churn_dataset-testing-master.csv")

display(df)

In [0]:
from pyspark.sql.functions import col

trusted_df = df.dropDuplicates()

trusted_df = trusted_df.fillna({
    "Age": 0
})

In [0]:
df_cleaned = df
for column in df.columns:
    df_cleaned = df_cleaned.withColumnRenamed(column, column.replace(" ", "_").lower())

df_cleaned.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "architect_company.churn_prediction.trusted_customers"
    )

In [0]:
from pyspark.sql.functions import avg

curated_df = trusted_df.groupBy("Age") \
    .agg(avg("Churn").alias("avg_churn"))

In [0]:
curated_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "architect_company.churn_prediction.trusted_customers_kpis"
    )

display(curated_df)